<a href="https://colab.research.google.com/github/fidlarsyn/grokking-Deep-Learning-Chapter-11-to-15/blob/main/12_Neural_Networks_That_Write_Like_Shakespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The Challenge of Arbitrary Length**

Mengembangkan pemahaman dari Bab 11 mengenai *word embeddings* tunggal menjadi pemodelan frasa dan kalimat dengan panjang variabel (*variable-length/arbitrary length sequences*). Tantangan terbesarnya adalah menciptakan vektor representasi kalimat berukuran tetap dari urutan kata yang panjangnya dapat berubah-ubah.

# **Do Comparisons Really Matter?**

Mengukur kemiripan antar-vektor kalimat sangat krusial karena vektor tersebut merupakan satu-satunya input yang dapat dilihat oleh jaringan saraf. Jika dua kalimat yang serupa secara makna tidak menghasilkan vektor yang mirip, jaringan saraf akan kesulitan menemukan korelasi antara representasi kalimat dan label target yang ingin diprediksi.

# **The Surprising Power of Averaged Word Vectors**

Metode intuitif pertama untuk membuat vektor kalimat adalah menghitung rata-rata (*average*) dari seluruh vektor kata yang ada di dalam kalimat tersebut. Pendekatan ini sangat ampuh karena aspek makna paling dominan dari kalimat akan bertahan, sedangkan *noise* dari kata-kata individual akan tereliminasi.

# **How Is Information Stored in These Embeddings? &amp; How Does a Neural Network Use Embeddings?**

Informasi disimpan di dalam *embeddings* sebagai pola kurva atau lonjakan (*bends/spikes*) variabel bobot. Jaringan saraf menggunakan representasi ini dengan mencari korelasi antara pola kurva *hidden layer* dengan target prediksi, sehingga kata-kata atau kalimat dengan aspek kemiripan serupa akan berbagi pola kurva yang mirip.

# **The Limitations of Bag-of-Words Vectors**

Metode penjumlahan atau rata-rata tergolong ke dalam pendekatan Bag-of-Words. Kelemahan utamanya adalah tidak memperhitungkan urutan kata (word order) karena sifat asosiatif penjumlahan ($a + b = b + a$). Mengacak urutan kata dalam kalimat akan menghasilkan vektor kalimat yang persis sama.

# **Using Identity Vectors to Sum Word Embeddings &amp; Learning the Transition Matrices**

 Untuk memasukkan informasi urutan, jaringan saraf diperkenalkan pada matriks identitas (identity matrix). Dengan mengganti matriks identitas menjadi matriks bobot yang dapat dilatih—disebut transition matrix atau recurrent matrix ($W_{rec}$)—jaringan saraf belajar memperbarui status memori kalimat setiap kali satu kata baru dibaca.

# **Learning to Create Useful Sentence Vectors &amp; Forward Propagation in Python**

Alur forward propagation pada Recurrent Neural Network (RNN) memproses sekuens kata secara bertahap pada setiap timestep $t$: $$\text{hidden}_t = \text{hidden}_{t-1} \cdot W_{rec} + \text{embed}[\text{word}_t]$$
 Status tersembunyi ($\text{hidden}_t$) berfungsi sebagai vektor memori yang membawa konteks kata-kata sebelumnya

In [ ]:
import sys, random, math
import numpy as np

np.random.seed(1)

# 1. Dataset QA Sederhana
lines = [
    "sandra moved to the garden",
    "sandra went back to the bedroom",
    "mary moved to the hallway",
    "daniel went back to the office"
]

tokens = [line.split(" ") for line in lines]

vocab = set()
for sent in tokens:
    for word in sent:
        vocab.add(word)
vocab = list(vocab)

word2index = {\}
for i, word in enumerate(vocab):
    word2index[word] = i

def words2indices(sentence):
    idx = list()
    for word in sentence:
        idx.append(word2index[word])
    return idx

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

# 2. Inisialisasi Parameter & Matriks Bobot RNN
embed_size = 10
embed = (np.random.rand(len(vocab), embed_size) - 0.5) * 0.1
recurrent = np.eye(embed_size)
start = np.zeros(embed_size)
decoder = (np.random.rand(embed_size, len(vocab)) - 0.5) * 0.1
one_hot = np.eye(len(vocab))

# 3. Fungsi Predict (Forward Propagation Sequence)
def predict(sent):
    layers = list()
    layer = {\}
    layer['hidden'] = start
    layers.append(layer)

    loss = 0
    preds = list()
    for target_i in range(len(sent)):
        layer = {\}
        # Prediksi kata berikutnya
        layer['pred'] = softmax(layers[-1]['hidden'].dot(decoder))
        loss += -np.log(layer['pred'][sent[target_i]])

        # Update Hidden State: (hidden_prev * recurrent) + current_word_embedding
        layer['hidden'] = layers[-1]['hidden'].dot(recurrent) + embed[sent[target_i]]
        layers.append(layer)

    return layers, loss

# 4. Training Loop (BPTT & Weight Update)
alpha = 0.001

for iter in range(30000):
    sent = words2indices(tokens[iter % len(tokens)])
    layers, loss = predict(sent)

    # Backpropagation Through Time (BPTT)
    for layer_idx in reversed(range(len(layers))):
        layer = layers[layer_idx]
        target = sent[layer_idx - 1] if layer_idx > 0 else 0

        if layer_idx > 0:
            layer['output_delta'] = layer['pred'] - one_hot[target]
            new_hidden_delta = layer['output_delta'].dot(decoder.T)

            if layer_idx == len(layers) - 1:
                layer['hidden_delta'] = new_hidden_delta
            else:
                layer['hidden_delta'] = new_hidden_delta + layers[layer_idx + 1]['hidden_delta'].dot(recurrent.T)
        else:
            layer['hidden_delta'] = layers[layer_idx + 1]['hidden_delta'].dot(recurrent.T)

    # Pembaruan Bobot (Weight Update)
    for layer_idx in range(1, len(layers)):
        layer = layers[layer_idx]
        target = sent[layer_idx - 1]

        decoder -= alpha * np.outer(layers[layer_idx - 1]['hidden'], layer['output_delta'])
        embed[target] -= alpha * layer['hidden_delta']
        recurrent -= alpha * np.outer(layers[layer_idx - 1]['hidden'], layer['hidden_delta'])

    if iter % 5000 == 0:
        perplexity = np.exp(loss / len(sent))
        print(f"Iterasi {iter:05d\} | Perplexity: {perplexity:.4f\}")

# **Backpropagation and Weight Update with Arbitrary Length**

Backpropagation pada sekuens panjang dilakukan secara mundur melewati seluruh timestep yang telah dilalui (dikenal sebagai Backpropagation Through Time / BPTT). Sinyal eror pada timestep saat ini ($\text{hidden\_delta}_t$) merambat mundur ke timestep sebelumnya melalui perkalian transpose matriks rekuren ($W_{rec}^T$).

# **Execution, Output Analysis, and Summary**

Evaluasi performa pelatihan RNN menggunakan metrik **perplexity** (eksponensial dari nilai *cross-entropy loss*) yang mengukur perbedaan antara distribusi probabilitas prediksi dengan label kata sebenarnya. Keterbatasan utama RNN linier dasar adalah kecenderungannya hanya mengingat beberapa kata terakhir dan memprediksi kata paling umum (seperti *"the"*). Tantangan ini menjadi landasan diperkenalkannya mekanisme pintu gerbang (*gating*) pada arsitektur **LSTM**.